# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their fields and column `@id`s.

In [ ]:
# List all record sets in the dataset with their @id
record_sets = list(dataset.record_sets)
print(f"Record sets found ({len(record_sets)}):")
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs.get('name', '')}")

# Display all fields and columns for each record set, by @id
for rs in record_sets:
    print(f"\nFields in record set '{rs['@id']}':")
    for field in rs.get('field', []):
        print(f"  • Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")
        column = field.get('column')
        # The field might directly reference one or multiple columns
        if column is not None:
            if isinstance(column, list):
                for col in column:
                    if isinstance(col, dict):
                        print(f"    - Column @id: {col.get('@id','')}, name: {col.get('name','')}, dataType: {col.get('dataType','')}")
                    else:
                        print(f"    - Column @id: {col}")
            elif isinstance(column, dict):
                print(f"    - Column @id: {column.get('@id','')}, name: {column.get('name','')}, dataType: {column.get('dataType','')}")
            else:
                print(f"    - Column @id: {column}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field references are via their `@id`.

In [ ]:
# List all record set @id's from above to use as variables
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

# For demonstration, load the first record set (change if needed)
example_record_set_id = record_set_ids[0]
print(f"Loading record set: {example_record_set_id}")

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns and first rows of the selected record set
print(f"\nColumn names in '{example_record_set_id}':\n{dataframes[example_record_set_id].columns.tolist()}")
dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All fields must be referenced by `@id`.

In [ ]:
# Example: Filter, normalize, and group data by column @id

# List fields and columns for the chosen record set (from Data Overview)
selected_df = dataframes[example_record_set_id]
print("Available fields:", selected_df.columns.tolist())

# Suppose there is a field '@id' for peptide count and one for protein name
# (adapt these to your actual field @id's from the overview; placeholder values are used below)
numeric_field_id = None
group_field_id = None

# Try to autodetect some sensible numeric and group fields for demonstration purposes
for col in selected_df.columns:
    if ('peptide' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or 'abundance' in col.lower()) and selected_df[col].dtype != 'O':
        numeric_field_id = col
    elif 'protein' in col.lower() or 'name' in col.lower() or 'accession' in col.lower():
        group_field_id = col

# Fallback if not found
if numeric_field_id is None:
    # Select first numeric column
    for col in selected_df.select_dtypes(include=['number']).columns:
        numeric_field_id = col
        break
if numeric_field_id is None:
    raise ValueError('No numeric field found for EDA in this record set!')

print(f"\nNumeric field selected for analysis (by @id): {numeric_field_id}")
if group_field_id:
    print(f"Group field (by @id): {group_field_id}")

# Filter rows based on value threshold
threshold = selected_df[numeric_field_id].quantile(0.75)
filtered_df = selected_df[selected_df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.3f}:")
print(filtered_df.head())

# Normalize
normalized_field = f"{numeric_field_id}_normalized"
filtered_df[normalized_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, normalized_field]].head())

# Group and aggregate if group field exists
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between the fields, using only `@id` references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(7, 4))
sns.histplot(selected_df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If group field exists, visualize means by group (only if number of groups is small)
if group_field_id and group_field_id in selected_df.columns:
    means = selected_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    # Plot the top 10 abundant groups
    plt.figure(figsize=(10, 5))
    means_sorted = means.sort_values(numeric_field_id, ascending=False).head(10)
    sns.barplot(y=group_field_id, x=numeric_field_id, data=means_sorted, palette="viridis")
    plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
With `mlcroissant`, we've loaded and explored the FAIR^2 dataset Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. The notebook demonstrates how to list record sets, display field and column `@id`s, extract data for analysis, filter and group by attributes using only `@id`, and visualize key numeric fields. Continue your own exploration by referencing any field or record set by its Croissant `@id`.